## Entity Resolution at Scale (TF-IDF / Cosine Similarity + Blocking)

Matching fragmented records of the same real-world entity (a client, a company) across naming variants — typos, abbreviations, legal-suffix differences — without exact-match or heavy embedding infrastructure.

**Why Character N-Gram TF-IDF, Not Word-Level or Exact Match**

1. TF-IDF(t,d) = tf(t,d) × log(N/df(t)) — term frequency downweighted by how common a term is across all documents.
2. Exact matching fails on naming variants; Levenshtein is character-level and fails on reordered tokens ("ABC Trading Co" vs. "Co ABC Trading"); embeddings are heavier to deploy for a problem that's mostly lexical overlap, not deep semantics.
3. Character n-grams, not word-level tokens, are the real upgrade — word-level treats a typo or abbreviation as an entirely different token, while character n-grams preserve partial overlap.
4. For entity names specifically, distinctive tokens (a rare surname, a company identifier) naturally carry more weight than common tokens ("Ltd", "Inc") purely from IDF downweighting — no extra logic needed to stop generic suffixes from dominating the match score.

In [ ]:
import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)
canonical_names = [
    "Meridian Trading Company Limited", "Alpine Logistics Group Pte Ltd", "Sundara Textiles Private Limited",
    "Orion Freight Solutions Inc", "Blackwood Commodities LLC", "Northgate Manufacturing Co",
    "Vantage Export Traders Ltd", "Sapphire Marine Shipping Corp", "Everline Industrial Supplies",
    "Coral Bay Import Export Ltd",
    # Genuinely confusable near-duplicates - distinct real entities, similarly named
    "Meridian Trading Corp", "Northgate Manufacturing Group",
]

def make_variant(name, rng):
    variant = name
    choice = rng.integers(0, 5)
    if choice == 0:
        variant = variant.replace("Limited", "Ltd").replace("Company", "Co").replace("Private", "Pvt")
    elif choice == 1:
        for suf in [" Limited", " Ltd", " Pte Ltd", " Inc", " LLC", " Corp", " Co"]:
            variant = variant.replace(suf, "")
    elif choice == 2:
        i = rng.integers(0, len(variant) - 1)
        variant = variant[:i] + variant[i + 1] + variant[i] + variant[i + 2:]
    elif choice == 3:
        variant = variant.replace(" ", "  ").strip() + "."
    return variant

records = []
for entity_id, name in enumerate(canonical_names):
    for _ in range(RNG.integers(2, 5)):
        records.append({"true_entity_id": entity_id, "record_name": make_variant(name, RNG)})
records_df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"{len(records_df)} fragmented records across {len(canonical_names)} true entities")

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), lowercase=True)
tfidf_matrix = vectorizer.fit_transform(records_df["record_name"])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape} (records x character n-gram vocabulary)")

**Cosine Similarity as the Match Score**

1. cos(θ) = (A·B) / (‖A‖‖B‖) — the angle between two TF-IDF vectors.
2. Normalized so it's robust to name-length differences (a long legal name vs. a short abbreviation), unlike raw Euclidean distance — a short abbreviated name shouldn't be penalized just for having fewer character n-grams overall.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

full_sim_matrix = cosine_similarity(tfidf_matrix)
same_entity_pair = records_df[records_df.true_entity_id == 0].index[:2]
diff_entity_pair = [records_df[records_df.true_entity_id == 0].index[0], records_df[records_df.true_entity_id == 1].index[0]]
print(f"Same-entity pair similarity: {full_sim_matrix[same_entity_pair[0], same_entity_pair[1]]:.3f}")
print(f"Different-entity pair similarity: {full_sim_matrix[diff_entity_pair[0], diff_entity_pair[1]]:.3f}")

**Blocking to Avoid O(n²) Comparisons**

1. Group candidate pairs by a coarse key first (country, registry-code prefix, first characters), then only compute full cosine similarity within blocks — cutting comparisons from all-pairs O(n²) down to a much smaller candidate set.
2. A first-2-characters key (used below for illustration) would miss true matches whose first characters differ due to a typo right at the start of the name — a real system blocks on something more stable, like a registry number prefix or normalized country code.

In [ ]:
records_df["block_key"] = records_df["record_name"].str.strip().str[:2].str.lower()
candidate_pairs = []
for block_key, group in records_df.groupby("block_key"):
    idxs = group.index.tolist()
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            candidate_pairs.append((idxs[i], idxs[j]))

total_pairs_no_blocking = len(records_df) * (len(records_df) - 1) // 2
print(f"All-pairs comparisons (no blocking): {total_pairs_no_blocking}")
print(f"Candidate pairs after blocking: {len(candidate_pairs)}")
print(f"Reduction: {(1 - len(candidate_pairs) / total_pairs_no_blocking):.1%}")

**Choosing the Match Threshold**

1. Sweep candidate thresholds against known true/false match labels, favoring precision.
2. A false merge corrupts a record permanently — worse than a missed match, which just stays unresolved for manual review.
3. Genuinely confusable near-duplicate entities (e.g. "Meridian Trading Corp" vs. "...Company Limited") make this a real precision/recall tradeoff, not a formality — precision visibly drops at low thresholds.

In [ ]:
pair_scores = []
for i, j in candidate_pairs:
    sim = full_sim_matrix[i, j]
    is_true_match = records_df.loc[i, "true_entity_id"] == records_df.loc[j, "true_entity_id"]
    pair_scores.append({"i": i, "j": j, "similarity": sim, "true_match": is_true_match})
pairs_df = pd.DataFrame(pair_scores)

from sklearn.metrics import precision_score, recall_score, f1_score
results = []
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = (pairs_df["similarity"] >= threshold).astype(int)
    true = pairs_df["true_match"].astype(int)
    results.append({"threshold": threshold, "precision": precision_score(true, pred, zero_division=0),
                     "recall": recall_score(true, pred, zero_division=0), "f1": f1_score(true, pred, zero_division=0)})
results_df = pd.DataFrame(results)
print(results_df.round(3))

qualifying = results_df[results_df["precision"] >= 0.90]
best_threshold = qualifying["threshold"].min() if len(qualifying) else results_df.loc[results_df["precision"].idxmax(), "threshold"]
pred_final = (pairs_df["similarity"] >= best_threshold).astype(int)
accuracy = (pred_final == pairs_df["true_match"].astype(int)).mean()
print(f"\nSelected threshold: {best_threshold}, overall accuracy at this threshold: {accuracy:.1%}")

**Handling the Borderline Band**

1. A false match merges two distinct entities — bad for downstream targeting and for compliance/KYC accuracy.
2. Borderline-similarity pairs get routed to manual review rather than auto-merged or auto-rejected — the borderline band genuinely mixes true and false matches, not cleanly separable at that similarity range, which is exactly why it needs a human decision rather than a lower automated threshold that would misclassify some fraction either direction.

In [ ]:
LOWER_BAND, UPPER_BAND = best_threshold - 0.15, best_threshold + 0.15
borderline = pairs_df[(pairs_df["similarity"] >= LOWER_BAND) & (pairs_df["similarity"] < UPPER_BAND)]
print(f"Borderline pairs routed to manual review ({LOWER_BAND:.2f}-{UPPER_BAND:.2f}): {len(borderline)}")
if len(borderline):
    print(borderline[["similarity", "true_match"]].sort_values("similarity", ascending=False))

**What's Still Missing for a Production Version**

- Real labeled validation set: this build's labels come from the synthetic generation process itself; production needs a manually reviewed sample.
- Better blocking key: first-2-characters is a toy key; production blocks on registration number, country + normalized prefix, or a locality-sensitive hash.
- Compliance constraints: a real entity-resolution merge has regulatory implications beyond match accuracy alone.
- Multi-lingual coverage: neither character n-gram TF-IDF nor generic phonetic encoding solves transliteration variants (e.g. pinyin) — cross-border names need transliteration-specific normalization, not a generic phonetic library.